<div style="direction: rtl; white-space: normal; line-height: 1;">
# Validate Quran Raw Data

هدف این مرحله:

- خواندن فایل عربی عثمان طه
- خواندن ترجمه فولادوند
- خواندن ترجمه انصاریان
- بررسی هماهنگی شماره سوره و آیه
- آماده‌سازی برای ساخت JSON آیه‌محور
</div>

In [1]:
from pathlib import Path
import json


# Project root
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


# Raw data
raw_dir = project_root / "data" / "raw"

arabic_file = raw_dir / "quran_arabic_uthmani.txt"
fooladvand_file = raw_dir / "quran_fa_fooladvand.txt"
ansarian_file = raw_dir / "quran_fa_ansarian.txt"


# Processed output
processed_dir = project_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)


json_output = processed_dir / "quran_dataset.json"


print("Project root:", project_root)
print("Raw directory:", raw_dir)
print("Output:", json_output)

Project root: /Users/macbookpro/Desktop/_PROGRAMING/QuranRAG
Raw directory: /Users/macbookpro/Desktop/_PROGRAMING/QuranRAG/data/raw
Output: /Users/macbookpro/Desktop/_PROGRAMING/QuranRAG/data/processed/quran_dataset.json


<div style="direction: rtl; white-space: normal; line-height: 1;">
خواندن سه فایل متنی
تبدیل هر فایل به دیکشنری با کلید:



مقایسه اینکه هر سه منبع آیه‌های یکسان دارند.
ساخت JSON نهایی.
</div>

In [3]:
# Read Quran text files

def read_quran_file(file_path):
    data = {}

    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            parts = line.split("|", 2)

            if len(parts) != 3:
                continue

            surah = int(parts[0])
            ayah = int(parts[1])
            text = parts[2]

            data[(surah, ayah)] = text

    return data


arabic_data = read_quran_file(arabic_file)
fooladvand_data = read_quran_file(fooladvand_file)
ansarian_data = read_quran_file(ansarian_file)


print("Arabic verses:", len(arabic_data))
print("Fooladvand verses:", len(fooladvand_data))
print("Ansarian verses:", len(ansarian_data))

Arabic verses: 6236
Fooladvand verses: 6236
Ansarian verses: 6236


#### ساخت JSON یکپارچه

In [4]:
# Build unified Quran dataset

quran_dataset = []

for key in arabic_data.keys():
    surah, ayah = key

    quran_dataset.append({
        "surah": surah,
        "ayah": ayah,
        "arabic": arabic_data[key],
        "fooladvand": fooladvand_data[key],
        "ansarian": ansarian_data[key]
    })


print("Total records:", len(quran_dataset))
print(quran_dataset[0])

Total records: 6236
{'surah': 1, 'ayah': 1, 'arabic': 'بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ', 'fooladvand': 'به نام خداوند رحمتگر مهربان', 'ansarian': 'به نام خدا که رحمتش بی\u200cاندازه است و مهربانی\u200cاش همیشگی.'}


ذخیره JSON

In [5]:
# Save Quran dataset as JSON

with open(json_output, "w", encoding="utf-8") as f:
    json.dump(
        quran_dataset,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Saved:", json_output)
print("Records:", len(quran_dataset))

Saved: /Users/macbookpro/Desktop/_PROGRAMING/QuranRAG/data/processed/quran_dataset.json
Records: 6236


تست کوچک بگیریم که فایل JSON

In [6]:
# Reload JSON test

with open(json_output, "r", encoding="utf-8") as f:
    test_data = json.load(f)


print("Loaded records:", len(test_data))
print(test_data[0])
print(test_data[-1])

Loaded records: 6236
{'surah': 1, 'ayah': 1, 'arabic': 'بِسْمِ ٱللَّهِ ٱلرَّحْمَٰنِ ٱلرَّحِيمِ', 'fooladvand': 'به نام خداوند رحمتگر مهربان', 'ansarian': 'به نام خدا که رحمتش بی\u200cاندازه است و مهربانی\u200cاش همیشگی.'}
{'surah': 114, 'ayah': 6, 'arabic': 'مِنَ ٱلْجِنَّةِ وَٱلنَّاسِ', 'fooladvand': 'چه از جنّ و [چه از] انس.»', 'ansarian': 'از جنّیان و آدمیان.'}
